##**1. Import Packages**

In [ ]:
# Import standard packages
import pandas as pd
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn import set_config
set_config(transform_output='pandas')

In [ ]:
# set the default output to pandas
from sklearn import set_config
set_config(transform_output='pandas')

##**2. Load Data**

In [ ]:
# Mount google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Load data from folder structure
fpath = '/content/drive/MyDrive/AXSOSACADEMY/02-IntroML/Week06/Data/medical_data.csv'
df = pd.read_csv(fpath)
df.head()

,State,Lat,Lng,Area,Children,Age,Income,Marital,Gender,ReAdmis,VitD_levels,Doc_visits,Full_meals_eaten,vitD_supp,Soft_drink,Initial_admin,HighBlood,Stroke,Complication_risk,Overweight,Arthritis,Diabetes,Hyperlipidemia,BackPain,Anxiety,Allergic_rhinitis,Reflux_esophagitis,Asthma,Services,Initial_days,TotalCharge,Additional_charges
0,AL,34.34960,-86.72508,Suburban,1.0,53,86575.93,Divorced,Male,0,19.141466,6,0,0,0,Emergency Admission,1,0,Medium,0,1.0,1.0,0.0,1.0,1.0,1.0,0,1,Blood Work,10.585770,3726.702860,17939.403420
1,FL,30.84513,-85.22907,Urban,3.0,51,46805.99,Married,Female,0,18.940352,4,2,1,0,Emergency Admission,1,0,High,1,0.0,0.0,0.0,0.0,0.0,0.0,1,0,Intravenous,15.129562,4193.190458,17612.998120
2,SD,43.54321,-96.63772,Suburban,3.0,53,14370.14,Widowed,Female,0,18.057507,4,1,0,0,Elective Admission,1,0,Medium,1,0.0,1.0,0.0,0.0,0.0,0.0,0,0,Blood Work,4.772177,2434.234222,17505.192460
3,MN,43.89744,-93.51479,Suburban,0.0,78,39741.49,Married,Male,0,16.576858,4,1,0,0,Elective Admission,0,1,Medium,0,1.0,0.0,0.0,0.0,0.0,0.0,1,1,Blood Work,1.714879,2127.830423,12993.437350
4,VA,37.59894,-76.88958,Rural,1.0,22,1209.56,Widowed,Female,0,17.439069,5,0,2,1,Elective Admission,0,0,Low,0,0.0,NaN,1.0,0.0,0.0,1.0,0,0,CT Scan,1.254807,2113.073274,3716.525786


##**3.Explore Data**

In [ ]:
# Check for duplicates
df.duplicated().sum()

np.int64(0)

In [ ]:
# Check data types and missing values
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 32 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   State               995 non-null    object 
 1   Lat                 1000 non-null   float64
 2   Lng                 1000 non-null   float64
 3   Area                995 non-null    object 
 4   Children            993 non-null    float64
 5   Age                 1000 non-null   int64  
 6   Income              1000 non-null   float64
 7   Marital             995 non-null    object 
 8   Gender              995 non-null    object 
 9   ReAdmis             1000 non-null   int64  
 10  VitD_levels         1000 non-null   float64
 11  Doc_visits          1000 non-null   int64  
 12  Full_meals_eaten    1000 non-null   int64  
 13  vitD_supp           1000 non-null   int64  
 14  Soft_drink          1000 non-null   int64  
 15  Initial_admin       995 non-null    object 
 16  HighBlo

In [ ]:
# Obtain summary statistics
df.describe()

,Lat,Lng,Children,Age,Income,ReAdmis,VitD_levels,Doc_visits,Full_meals_eaten,vitD_supp,Soft_drink,HighBlood,Stroke,Overweight,Arthritis,Diabetes,Hyperlipidemia,BackPain,Anxiety,Allergic_rhinitis,Reflux_esophagitis,Asthma,Initial_days,TotalCharge,Additional_charges
count,1000.000000,1000.000000,993.000000,1000.000000,1000.000000,1000.0,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,994.000000,994.00000,998.000000,992.000000,998.000000,994.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,38.989277,-91.165542,2.103726,54.084000,40653.044950,0.0,17.992381,4.992000,1.024000,0.366000,0.250000,0.415000,0.194000,0.716000,0.358149,0.28169,0.346693,0.414315,0.326653,0.396378,0.422000,0.308000,9.267176,3240.971613,13124.934863
std,5.504177,15.451957,2.239293,20.903203,28370.102213,0.0,2.056366,1.048349,1.013139,0.598667,0.433229,0.492969,0.395627,0.451162,0.479698,0.45005,0.476156,0.492852,0.469224,0.489391,0.494126,0.461898,6.030931,600.413722,6677.691402
min,18.010230,-171.688150,0.000000,18.000000,154.080000,0.0,11.475314,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.012586,2022.650007,3241.339760
25%,35.673860,-96.840598,0.000000,36.750000,19295.567500,0.0,16.620469,4.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,4.518085,2822.108670,8121.383834
50%,39.524835,-88.242890,1.000000,55.000000,34222.550000,0.0,18.020163,5.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,8.042516,3164.679246,11698.462430
75%,42.130840,-80.602785,3.000000,72.000000,54924.115000,0.0,19.418254,6.000000,2.000000,1.000000,0.250000,1.000000,0.000000,1.000000,1.000000,1.00000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,12.711671,3628.550654,16493.908180
max,70.560990,-66.247510,10.000000,89.000000,204542.410000,0.0,24.565463,9.000000,7.000000,3.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.00000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,35.269743,5572.846022,30087.650940


In [ ]:
# Summary for object columns
df.describe(include='object')

,State,Area,Marital,Gender,Initial_admin,Complication_risk,Services
count,995,995,995,995,995,995,995
unique,51,3,5,8,3,4,4
top,TX,Rural,Divorced,Female,Emergency Admission,Medium,Blood Work
freq,54,346,207,513,502,459,524


In [ ]:
# Drop State
droplist = ['State']
df = df.drop(droplist, axis=1)

In [ ]:
# Explore Gender
df['Gender'].value_counts()

,count
Gender,
Female,513
Male,452
Nonbinary,21
m,3
male,2
F,2
f,1
M,1


In [ ]:
# Correct inconsistencies in values in Gender column
df['Gender'] = df['Gender'].replace(['male', 'm', 'M'], 'Male')
df['Gender'] = df['Gender'].replace(['F', 'f'], 'Female')
df['Gender'].value_counts()

,count
Gender,
Female,516
Male,458
Nonbinary,21


##**4. Perform a Validation Split**

In [ ]:
# Define X and y
X = df.drop(columns='Additional_charges')
y = df['Additional_charges']
# Split the data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

##**5. Create a ColumnTransformer for Preprocessing**

We used make_column_selector to intelligently define categorical columns.

We'll use it later with ColumnTransformer inside Pipeline.


The goal:

 to handle each data type correctly without the hassle of manually updating lists.

In [ ]:
# Using select_dtypes for categorical data
list(X_train.select_dtypes('object').columns)

['Area', 'Marital', 'Gender', 'Initial_admin', 'Complication_risk', 'Services']

In [ ]:
# Creating a "column selector" from sklearn
cat_selector = make_column_selector(dtype_include='object')
# Works just like select_dtypes from pandas
cat_selector(X_train)

['Area', 'Marital', 'Gender', 'Initial_admin', 'Complication_risk', 'Services']

##**Pipeline for Preprocessing Categorical Data**

In [ ]:
# Create the preprocessing pipeline for categorical data
# (New) Select columns with make_column_selector
cat_selector = make_column_selector(dtype_include='object')
# Insantiate transfomers
freq_imputer = SimpleImputer(strategy='most_frequent')
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
# Instantiate the pipeline
cat_pipe = make_pipeline(freq_imputer, ohe)
# Make a tuple for column transformer
cat_tuple = ('categorical',cat_pipe, cat_selector)
cat_pipe

Pipeline(steps=[('simpleimputer', SimpleImputer(strategy='most_frequent')),
                ('onehotencoder',
                 OneHotEncoder(handle_unknown='ignore', sparse_output=False))])

##**Pipeline for Preprocessing Numeric Data**

In [ ]:
# Create the preprocessing pipeline for numeric data
# (New) Select columns wiht make_column)selector
num_selector = make_column_selector(dtype_include='number')
# Instantiate the transformers
scaler = StandardScaler()
mean_imputer = SimpleImputer(strategy='mean')
# Instantiate the pipeline
num_pipe = make_pipeline(mean_imputer, scaler)
# Make the tuple for ColumnTransformer
num_tuple = ('numeric',num_pipe, num_selector)
num_pipe

Pipeline(steps=[('simpleimputer', SimpleImputer()),
                ('standardscaler', StandardScaler())])

##**Create the ColumnTranformer**

In [ ]:
# Create the preprocessing ColumnTransformer
preprocessor = ColumnTransformer([cat_tuple, num_tuple],
                                 verbose_feature_names_out=False)
preprocessor

ColumnTransformer(transformers=[('categorical',
                                 Pipeline(steps=[('simpleimputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehotencoder',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 <sklearn.compose._column_transformer.make_column_selector object at 0x7d7138f33c80>),
                                ('numeric',
                                 Pipeline(steps=[('simpleimputer',
                                                  SimpleImputer()),
                                                 ('standardscaler',
                                                  StandardScaler())]),
                                 <sklearn.compose._column_transformer.make_column_selector object at 0x7d7138f30aa0>)],
                  verbose_feature_names_out=False)

##**6. Create a Model Pipeline**

In [ ]:
# Instantiate a linear regression model
linreg = LinearRegression()
# Combine the preprocessing ColumnTransformer and the linear regression model in a Pipeline
linreg_pipe = make_pipeline(preprocessor, linreg)
linreg_pipe

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('categorical',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehotencoder',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x7d7138f33c80>),
                                                 ('numeric',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer()),
                                                                  ('standardscaler',
                                                                   StandardScaler())]),
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x7d7138f30aa0>)],
                                   verbose_feature_names_out=False)),
                ('linearregression', LinearRegression())])

##**7. Fit the Model Pipeline on the Training Data**

In [ ]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 750 entries, 82 to 102
Data columns (total 30 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Lat                 750 non-null    float64
 1   Lng                 750 non-null    float64
 2   Area                747 non-null    object 
 3   Children            746 non-null    float64
 4   Age                 750 non-null    int64  
 5   Income              750 non-null    float64
 6   Marital             747 non-null    object 
 7   Gender              746 non-null    object 
 8   ReAdmis             750 non-null    int64  
 9   VitD_levels         750 non-null    float64
 10  Doc_visits          750 non-null    int64  
 11  Full_meals_eaten    750 non-null    int64  
 12  vitD_supp           750 non-null    int64  
 13  Soft_drink          750 non-null    int64  
 14  Initial_admin       747 non-null    object 
 15  HighBlood           750 non-null    int64  
 16  Stroke      

In [ ]:
# Fit the model pipeline on the training data
linreg_pipe.fit(X_train, y_train)

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('categorical',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehotencoder',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x7d7138f33c80>),
                                                 ('numeric',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer()),
                                                                  ('standardscaler',
                                                                   StandardScaler())]),
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x7d7138f30aa0>)],
                                   verbose_feature_names_out=False)),
                ('linearregression', LinearRegression())])

##**8) Evaluate the Model**

In [ ]:
def regression_metrics(y_true, y_pred, label='', verbose = True, output_dict=False):
  # Get metrics
  mae = mean_absolute_error(y_true, y_pred)
  mse = mean_squared_error(y_true, y_pred)
  rmse = mean_squared_error(y_true, y_pred)
  r_squared = r2_score(y_true, y_pred)
  if verbose == True:
    # Print Result with Label and Header
    header = "-"*60
    print(header, f"Regression Metrics: {label}", header, sep='\n')
    print(f"- MAE = {mae:,.3f}")
    print(f"- MSE = {mse:,.3f}")
    print(f"- RMSE = {rmse:,.3f}")
    print(f"- R^2 = {r_squared:,.3f}")
  if output_dict == True:
      metrics = {'Label':label, 'MAE':mae,
                 'MSE':mse, 'RMSE':rmse, 'R^2':r_squared}
      return metrics

def evaluate_regression(reg, X_train, y_train, X_test, y_test, verbose = True,
                        output_frame=False):
  # Get predictions for training data
  y_train_pred = reg.predict(X_train)

  # Call the helper function to obtain regression metrics for training data
  results_train = regression_metrics(y_train, y_train_pred, verbose = verbose,
                                     output_dict=output_frame,
                                     label='Training Data')
  print()
  # Get predictions for test data
  y_test_pred = reg.predict(X_test)
  # Call the helper function to obtain regression metrics for test data
  results_test = regression_metrics(y_test, y_test_pred, verbose = verbose,
                                  output_dict=output_frame,
                                    label='Test Data' )

  # Store results in a dataframe if ouput_frame is True
  if output_frame:
    results_df = pd.DataFrame([results_train,results_test])
    # Set the label as the index
    results_df = results_df.set_index('Label')
    # Set index.name to none to get a cleaner looking result
    results_df.index.name=None
    # Return the dataframe
    return results_df.round(3)

In [ ]:
# Obtain Model Evulation using custom function
evaluate_regression(linreg_pipe, X_train, y_train, X_test, y_test)

------------------------------------------------------------
Regression Metrics: Training Data
------------------------------------------------------------
- MAE = 1,369.194
- MSE = 2,604,536.843
- RMSE = 2,604,536.843
- R^2 = 0.943

------------------------------------------------------------
Regression Metrics: Test Data
------------------------------------------------------------
- MAE = 1,362.389
- MSE = 2,755,824.958
- RMSE = 2,755,824.958
- R^2 = 0.933
